In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D11 — Investor Presentation Q4 & FY24
# ============================================================

!pip install pymupdf -q

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 16.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D11"

DOCUMENT_NAME = (
    "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24"
)

SOURCE_FORMAT = "PDF"

EXPECTED_PAGE_COUNT = 10

EXPECTED_REFERENCE_RECORD_COUNT = 199

REFERENCE_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
}

OUTPUT_DIR = Path("outputs_D11_stage1")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Document:", DOCUMENT_ID)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print(
    "Expected reference records:",
    EXPECTED_REFERENCE_RECORD_COUNT
)
print("Output directory:", OUTPUT_DIR)

Document: D11
Expected pages: 10
Expected reference records: 199
Output directory: outputs_D11_stage1


In [3]:
# ============================================================
# 2. Upload original PDF
# ============================================================

print(
    "Upload the original D11 investor-presentation PDF."
)

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError(
        "Upload exactly one PDF file."
    )

SOURCE_PATH = pdf_files[0]

print(
    "Loaded:",
    SOURCE_PATH.name
)

Upload the original D11 investor-presentation PDF.


Saving D11 - Investor-Presentation-May-2024.pdf to D11 - Investor-Presentation-May-2024.pdf
Loaded: D11 - Investor-Presentation-May-2024.pdf


In [4]:
# ============================================================
# 3. Source-file hashing
# ============================================================

def sha256_file(path):
    """
    Return the SHA-256 hash of a file.
    """

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):
            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

print(
    "Source SHA-256:",
    SOURCE_SHA256
)

Source SHA-256: 604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952


In [5]:
# ============================================================
# 4. Load PDF and extract native text
# ============================================================

document = fitz.open(
    SOURCE_PATH
)

PAGE_COUNT = len(
    document
)

PAGE_COUNT_VALID = (
    PAGE_COUNT
    == EXPECTED_PAGE_COUNT
)

page_texts = []

for page_index, page in enumerate(
    document,
    start=1
):

    page_text = page.get_text(
        "text"
    ) or ""

    page_texts.append({
        "Page Number":
            page_index,

        "Text":
            page_text
    })


FULL_TEXT = "\n".join(
    item["Text"]
    for item in page_texts
)

TEXT_EXTRACTABLE = bool(
    FULL_TEXT.strip()
)

OCR_REQUIRED = not TEXT_EXTRACTABLE


print(
    "Pages:",
    PAGE_COUNT
)

print(
    "Page count valid:",
    PAGE_COUNT_VALID
)

print(
    "Text extractable:",
    TEXT_EXTRACTABLE
)

print(
    "OCR required:",
    OCR_REQUIRED
)


if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )

if not TEXT_EXTRACTABLE:
    raise AssertionError(
        "The D11 PDF does not contain a usable "
        "native text layer."
    )

Pages: 10
Page count valid: True
Text extractable: True
OCR required: False


In [6]:
# ============================================================
# 5. Document metadata
# ============================================================

DOCUMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "physical_page_count":
        PAGE_COUNT,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "page_count_valid":
        bool(
            PAGE_COUNT_VALID
        ),

    "text_extractable":
        bool(
            TEXT_EXTRACTABLE
        ),

    "ocr_required":
        bool(
            OCR_REQUIRED
        ),

    "native_text_character_count":
        len(
            FULL_TEXT
        ),

    "native_text_word_count":
        len(
            FULL_TEXT.split()
        ),

}


print(
    json.dumps(
        DOCUMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D11",
  "document_name": "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "source_file_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "source_format": "PDF",
  "physical_page_count": 10,
  "expected_page_count": 10,
  "page_count_valid": true,
  "text_extractable": true,
  "ocr_required": false,
  "native_text_character_count": 8545,
  "native_text_word_count": 1332
}


In [7]:
# ============================================================
# 6. Document characterisation
# ============================================================

numeric_tokens = re.findall(
    r"(?<!\w)[+-]?\d[\d,]*(?:\.\d+)?",
    FULL_TEXT
)

percentage_tokens = re.findall(
    r"[+-]?\d+(?:\.\d+)?\s*%",
    FULL_TEXT
)

currency_tokens = re.findall(
    r"(?:Rs\.?|₹)\s*[+-]?\d[\d,]*(?:\.\d+)?",
    FULL_TEXT,
    flags=re.IGNORECASE
)

word_count = len(
    FULL_TEXT.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)


DOCUMENT_CHARACTERISATION = {
    "document_id":
        DOCUMENT_ID,

    "document_type":
        "Investor presentation",

    "source_format":
        SOURCE_FORMAT,

    "physical_page_count":
        PAGE_COUNT,

    "text_extractable":
        bool(
            TEXT_EXTRACTABLE
        ),

    "ocr_required":
        bool(
            OCR_REQUIRED
        ),

    "native_text_character_count":
        len(
            FULL_TEXT
        ),

    "native_text_word_count":
        word_count,

    "numeric_token_count":
        len(
            numeric_tokens
        ),

    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),

    "percentage_token_count":
        len(
            percentage_tokens
        ),

    "currency_token_count":
        len(
            currency_tokens
        ),

    "contains_safe_harbor":
        "Safe Harbor"
        in FULL_TEXT,

    "contains_management_commentary":
        "Management Commentary"
        in FULL_TEXT,

    "contains_financial_charts":
        "Quarterly Business Performance"
        in FULL_TEXT,

    "contains_financial_table":
        "Profit & Loss Statement"
        in FULL_TEXT,

    "contains_company_profile":
        "Our Legacy, Our Future"
        in FULL_TEXT,

    "contains_corporate_timeline":
        "We Improve. Grow. Accelerate"
        in FULL_TEXT,

    "contains_operational_infographic":
        "We serve multiple end markets"
        in FULL_TEXT,

    "contains_images_and_infographics":
        True,

    "contains_mixed_units":
        True,

    "contains_repeated_financial_metrics":
        True,

    "contains_qualified_operational_values":
        True,

    "detected_fiscal_periods":
        sorted(
            set(
                re.findall(
                    r"\b(?:Q[1-4]\s*)?FY\d{2}\b",
                    FULL_TEXT
                )
            )
        ),

    "detected_calendar_years":
        sorted(
            set(
                re.findall(
                    r"\b(?:19|20)\d{2}\b",
                    FULL_TEXT
                )
            )
        ),

    "main_information_regions": [
        "Management commentary",
        "Quarterly business-performance charts",
        "Q4FY24 Profit & Loss Statement",
        "Company-profile information",
        "Corporate timeline",
        "Operational-footprint infographic"
    ],

    "excluded_from_reference_scope": [
        "Purely decorative imagery",
        "Section-divider slides without extractable task records",
        "Forward-looking Safe Harbor prose beyond the selected metadata record",
        "Values that would require visual estimation rather than printed numerical evidence"
    ]
}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D11",
  "document_type": "Investor presentation",
  "source_format": "PDF",
  "physical_page_count": 10,
  "text_extractable": true,
  "ocr_required": false,
  "native_text_character_count": 8545,
  "native_text_word_count": 1332,
  "numeric_token_count": 196,
  "numeric_token_to_word_ratio": 0.147,
  "percentage_token_count": 42,
  "currency_token_count": 10,
  "contains_safe_harbor": true,
  "contains_management_commentary": true,
  "contains_financial_charts": true,
  "contains_financial_table": true,
  "contains_company_profile": true,
  "contains_corporate_timeline": true,
  "contains_operational_infographic": true,
  "contains_images_and_infographics": true,
  "contains_mixed_units": true,
  "contains_repeated_financial_metrics": true,
  "contains_qualified_operational_values": true,
  "detected_fiscal_periods": [
    "FY22",
    "FY23",
    "FY24",
    "Q3 FY24",
    "Q3FY24",
    "Q4 FY23",
    "Q4 FY24",
    "Q4FY23",
    "Q4FY24"
  ],
  "detected_calendar_

In [8]:
# ============================================================
# 7. Page-level characterisation
# ============================================================

page_characterisation = []

for item in page_texts:

    page_number = item[
        "Page Number"
    ]

    text = item[
        "Text"
    ]

    page_characterisation.append({
        "Physical PDF Page":
            page_number,

        "Text Characters":
            len(
                text
            ),

        "Word Count":
            len(
                text.split()
            ),

        "Numeric Token Count":
            len(
                re.findall(
                    r"(?<!\w)[+-]?\d[\d,]*(?:\.\d+)?",
                    text
                )
            ),

        "Percentage Token Count":
            len(
                re.findall(
                    r"[+-]?\d+(?:\.\d+)?\s*%",
                    text
                )
            ),

        "Currency Token Count":
            len(
                re.findall(
                    r"(?:Rs\.?|₹)\s*[+-]?\d[\d,]*(?:\.\d+)?",
                    text,
                    flags=re.IGNORECASE
                )
            ),

        "Contains Safe Harbor":
            "Safe Harbor"
            in text,

        "Contains Management Commentary":
            "Management Commentary"
            in text,

        "Contains Quarterly Performance":
            "Quarterly Business Performance"
            in text,

        "Contains P&L":
            "Profit & Loss Statement"
            in text,

        "Contains Company Profile":
            "Our Legacy, Our Future"
            in text,

        "Contains Corporate Timeline":
            "We Improve. Grow. Accelerate"
            in text,

        "Contains Operational Metrics":
            "We serve multiple end markets"
            in text
    })


page_characterisation_df = pd.DataFrame(
    page_characterisation
)

display(
    page_characterisation_df
)

,Physical PDF Page,Text Characters,Word Count,Numeric Token Count,Percentage Token Count,Currency Token Count,Contains Safe Harbor,Contains Management Commentary,Contains Quarterly Performance,Contains P&L,Contains Company Profile,Contains Corporate Timeline,Contains Operational Metrics
0,1,34,6,0,0,0,False,False,False,False,False,False,False
1,2,2749,398,1,0,0,True,False,False,False,False,False,False
2,3,22,4,0,0,0,False,False,False,False,False,False,False
3,4,1859,293,18,6,8,False,True,False,False,False,False,False
4,5,497,95,46,0,0,False,False,True,False,False,False,False
5,6,1063,188,103,35,2,False,False,False,True,False,False,False
6,7,23,4,0,0,0,False,False,False,False,True,False,False
7,8,778,112,5,0,0,False,False,False,False,True,False,False
8,9,1193,187,16,1,0,False,False,False,False,False,True,False
9,10,318,45,7,0,0,False,False,False,False,False,False,True


In [9]:
# ============================================================
# 8. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed set of financial, corporate-profile,
historical and operational records represented in the
Siyaram Silk Mills Limited Investor Presentation Q4 & FY24.

Include exactly the predefined records belonging to:

- Presentation metadata
- Management commentary
- Quarterly business performance
- Profit and loss statement
- Company profile
- Corporate timeline
- Operational footprint

For every record return:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Use only information explicitly represented in the supplied document.
- Preserve repeated observations when the same metric appears in
  different source sections or representations.
- Do not deduplicate management-commentary, chart and P&L observations
  solely because the reported value is the same.
- Extract chart values only when a numerical value is explicitly printed.
- Do not visually estimate values from graphical size or position.
- Preserve the association between quarterly chart segments, fiscal
  years and annual totals.
- Do not interpret blank YoY or QoQ table cells as numerical values.
- Preserve approximation and lower-bound wording such as "~", "+"
  and "and counting".
- Preserve the printed measurement scale and unit.
- Do not calculate, infer, reconstruct, derive, convert or correct values.
- Keep differently scoped observations separate, including the
  247-store observation in management commentary and the separately
  represented 245+ store estimate in the operational infographic.
- Return exactly 199 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""

print(
    EXTRACTION_TASK
)


Extract the fixed set of financial, corporate-profile,
historical and operational records represented in the
Siyaram Silk Mills Limited Investor Presentation Q4 & FY24.

Include exactly the predefined records belonging to:

- Presentation metadata
- Management commentary
- Quarterly business performance
- Profit and loss statement
- Company profile
- Corporate timeline
- Operational footprint

For every record return:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Rules:

- Use only information explicitly represented in the supplied document.
- Preserve repeated observations when the same metric appears in
  different source sections or representations.
- Do not deduplicate management-commentary, chart and P&L observations
  solely because the reported value is the same.
- Extract chart values only when a numerical value is explicitly printed.
- Do not visually estimate values from graphical size or position.
- Preserve the association between q

In [10]:
# ============================================================
# 9. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "investor_presentation_item",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": [
                "string",
                "null"
            ]
        },

        "Topic": {
            "type": [
                "string",
                "null"
            ]
        },

        "Description": {
            "type": [
                "string",
                "null"
            ]
        },

        "Value": {
            "type": [
                "string",
                "number",
                "null"
            ]
        },

        "Unit": {
            "type": [
                "string",
                "null"
            ]
        },

        "Reporting Period": {
            "type": [
                "string",
                "null"
            ]
        },

        "Source Location": {
            "type": [
                "string",
                "null"
            ]
        }
    },

    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,

        "records": [
            {
                "Category":
                    "string or null",

                "Topic":
                    "string or null",

                "Description":
                    "string or null",

                "Value":
                    "string, number or null",

                "Unit":
                    "string or null",

                "Reporting Period":
                    "string or null",

                "Source Location":
                    "string or null"
            }
        ]
    }
}


print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D11",
  "record_level": "investor_presentation_item",
  "expected_record_count": 199,
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Topic": {
      "type": [
        "string",
        "null"
      ]
    },
    "Description": {
      "type": [
        "string",
        "null"
      ]
    },
    "Value": {
      "type": [
        "string",
        "number",
        "null"
      ]
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ]
    },
    "Reporting Period": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D11",
    "records": [
      {
        "Category": "string or null",
        "Topic": "string or null",
        "Description": "string or null",
        "Value": "string, number or null",
        "Unit": "string or null",
 

In [11]:
# ============================================================
# 10. Fixed reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id":
        DOCUMENT_ID,

    "record_level":
        "investor_presentation_item",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "fields": {
        "Category":
            "Fixed D11 reference-record category.",

        "Topic":
            "Source-grounded metric, concept, statement or event.",

        "Description":
            "Source-grounded description of the represented record.",

        "Value":
            (
                "Explicit numeric or textual value. "
                "Qualified values preserve their printed qualifier."
            ),

        "Unit":
            "Printed or contextually attached measurement unit.",

        "Reporting Period":
            (
                "Quarter, fiscal year, date or historical interval "
                "explicitly associated with the record."
            ),

        "Source Location":
            "Physical PDF page supporting the reference record."
    },

    "value_rules": {
        "explicit_exact_number":
            "Stored as a JSON number.",

        "qualified_number":
            (
                "Stored as source-grounded text preserving "
                "~, + or equivalent qualifier."
            ),

        "qualitative_value":
            "Stored as source-grounded text.",

        "unsupported_value":
            "Not inferred or reconstructed."
    },

    "reference_construction_rules": [
        (
            "Repeated metrics represented in different document "
            "regions remain separate reference records."
        ),
        (
            "Printed chart values are used; values are not "
            "estimated visually from chart geometry."
        ),
        (
            "Blank YoY and QoQ P&L cells are not represented "
            "as inferred numerical observations."
        ),
        (
            "Qualified operational metrics preserve source "
            "markers such as ~, + and 'and counting'."
        ),
        (
            "Differently scoped store-count observations are "
            "preserved separately rather than reconciled."
        )
    ],

    "branch_reuse":
        (
            "The same fixed reference dataset is reused for "
            "Branches A, B and C."
        )
}


print(
    json.dumps(
        REFERENCE_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D11",
  "record_level": "investor_presentation_item",
  "expected_record_count": 199,
  "expected_category_counts": {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
  },
  "fields": {
    "Category": "Fixed D11 reference-record category.",
    "Topic": "Source-grounded metric, concept, statement or event.",
    "Description": "Source-grounded description of the represented record.",
    "Value": "Explicit numeric or textual value. Qualified values preserve their printed qualifier.",
    "Unit": "Printed or contextually attached measurement unit.",
    "Reporting Period": "Quarter, fiscal year, date or historical interval explicitly associated with the record.",
    "Source Location": "Physical PDF page supporting the reference record."
  },
  "value_rules": {
    "explicit_exact_numbe

In [12]:
# ============================================================
# 11. Independent reference dataset
# ============================================================

records = []

def add_record(category, topic, description, value, unit, period, page):
    records.append({
        "Category": category,
        "Topic": topic,
        "Description": description,
        "Value": value,
        "Unit": unit,
        "Reporting Period": period,
        "Source Location": f"PDF page {page}"
    })

# ------------------------------------------------------------------
# Presentation metadata — pages 1–2
# ------------------------------------------------------------------
add_record(
    "Presentation metadata",
    "Presentation title",
    "Title of the investor presentation",
    "Investor Presentation | Q4 & FY24",
    "text",
    "Q4 & FY24",
    1
)
add_record(
    "Presentation metadata",
    "Company name",
    "Company identified as the preparer of the presentation",
    "Siyaram Silk Mills Limited",
    "text",
    "Q4 & FY24",
    2
)
add_record(
    "Presentation metadata",
    "Safe Harbor",
    "Presentation prepared solely for information purposes and not as an offer, recommendation or invitation to purchase or subscribe for securities",
    "Safe Harbor statement",
    "text",
    "Q4 & FY24",
    2
)


# ------------------------------------------------------------------
# Management commentary — page 4
# ------------------------------------------------------------------
management_records = [
    ("Market conditions",
     "Market conditions faced by the company",
     "Subdued consumer demand and challenging market conditions",
     "text", "Q4 FY24"),

    ("Revenue from Operations",
     "Revenue from Operations reported in management commentary",
     6464, "Rs. Mn", "Q4 FY24"),

    ("Comparative Revenue from Operations",
     "Comparative Revenue from Operations reported in management commentary",
     6948, "Rs. Mn", "Q4 FY23"),

    ("Revenue mix — Fabric",
     "Fabric share of the Q4 FY24 revenue mix",
     82, "percent", "Q4 FY24"),

    ("Revenue mix — Garments",
     "Garments share of the Q4 FY24 revenue mix",
     13, "percent", "Q4 FY24"),

    ("Revenue mix — Yarn & Others",
     "Yarn & Others share of the Q4 FY24 revenue mix",
     5, "percent", "Q4 FY24"),

    ("EBITDA",
     "EBITDA reported in management commentary",
     1059, "Rs. Mn", "Q4 FY24"),

    ("EBITDA Margin",
     "EBITDA margin reported in management commentary",
     16.4, "percent", "Q4 FY24"),

    ("Profit After Tax",
     "Profit After Tax reported in management commentary",
     690, "Rs. Mn", "Q4 FY24"),

    ("PAT Margin",
     "PAT margin reported in management commentary",
     10.7, "percent", "Q4 FY24"),

    ("Retail footprint",
     "Total retail footprint as of March 31, 2024",
     247, "stores", "2024-03-31"),

    ("Sales promotion spending",
     "Sales promotion spending allocated in Q4 FY24",
     12.04, "Rs. crores", "Q4 FY24"),

    ("Comparative sales promotion spending",
     "Sales promotion spending reported for Q3 FY24",
     17.5, "Rs. crores", "Q3 FY24"),

    ("Dividend",
     "Dividend approved by the board of directors",
     4, "Rs. per share", "FY24"),

    ("Dividend percentage",
     "Dividend as a percentage of face value",
     200, "percent", "FY24"),

    ("Face value",
     "Face value of each share used for the dividend statement",
     2, "Rs. per share", "FY24"),

    ("Management spokesperson",
     "Executive Director named in the management commentary",
     "Mr. Gaurav Poddar",
     "text", "Q4 FY24")
]

for topic, description, value, unit, period in management_records:
    add_record(
        "Management commentary",
        topic,
        description,
        value,
        unit,
        period,
        4
    )


# ------------------------------------------------------------------
# Quarterly Business Performance — page 5
# ------------------------------------------------------------------
quarterly_performance = {
    "Net Revenue": {
        "description": "Net Revenue excluding Other Income; standalone financials; values rounded to the nearest whole number",
        "annual": {"FY22": 19031, "FY23": 22293, "FY24": 20872},
        "quarterly": {
            "Q1 FY22": 2327, "Q2 FY22": 4799, "Q3 FY22": 5625, "Q4 FY22": 6280,
            "Q1 FY23": 3980, "Q2 FY23": 6355, "Q3 FY23": 5011, "Q4 FY23": 6948,
            "Q1 FY24": 3537, "Q2 FY24": 5852, "Q3 FY24": 5019, "Q4 FY24": 6464
        }
    },
    "EBITDA": {
        "description": "EBITDA excluding Other Income; standalone financials; values rounded to the nearest whole number",
        "annual": {"FY22": 3343, "FY23": 3689, "FY24": 2849},
        "quarterly": {
            "Q1 FY22": 295, "Q2 FY22": 849, "Q3 FY22": 1021, "Q4 FY22": 1177,
            "Q1 FY23": 519, "Q2 FY23": 1198, "Q3 FY23": 758, "Q4 FY23": 1215,
            "Q1 FY24": 225, "Q2 FY24": 880, "Q3 FY24": 685, "Q4 FY24": 1059
        }
    },
    "Net Profit After Tax": {
        "description": "Net Profit After Tax; standalone financials; values rounded to the nearest whole number",
        "annual": {"FY22": 2125, "FY23": 2518, "FY24": 1847},
        "quarterly": {
            "Q1 FY22": 130, "Q2 FY22": 533, "Q3 FY22": 690, "Q4 FY22": 773,
            "Q1 FY23": 310, "Q2 FY23": 806, "Q3 FY23": 520, "Q4 FY23": 883,
            "Q1 FY24": 100, "Q2 FY24": 614, "Q3 FY24": 443, "Q4 FY24": 690
        }
    }
}

for topic, metric_data in quarterly_performance.items():
    for period, value in metric_data["annual"].items():
        add_record(
            "Quarterly business performance",
            topic,
            metric_data["description"] + "; annual total",
            value,
            "Rs. Mn",
            period,
            5
        )

    for period, value in metric_data["quarterly"].items():
        add_record(
            "Quarterly business performance",
            topic,
            metric_data["description"] + "; quarterly component",
            value,
            "Rs. Mn",
            period,
            5
        )


# ------------------------------------------------------------------
# Q4 FY24 Profit & Loss Statement — page 6
# ------------------------------------------------------------------
period_columns = [
    ("Q4 FY24", "Q4 FY24"),
    ("Q4 FY23", "Q4 FY23"),
    ("Q3 FY24", "Q3 FY24"),
    ("FY24", "FY24"),
    ("FY23", "FY23")
]

pnl_exact_values = {
    "Revenue from Operations": [6464, 6948, 5019, 20872, 22293],
    "Cost Of Goods Sold": [3863, 4234, 2879, 12106, 12971],
    "Employee Expenses": [450, 511, 429, 1723, 1791],
    "Other Expenses": [1092, 988, 1026, 4194, 3842],
    "EBITDA": [1059, 1215, 685, 2849, 3689],
    "EBITDA Margin": [16.4, 17.5, 13.6, 13.6, 16.5],
    "Other Income": [64, 103, 111, 376, 402],
    "Depreciation": [140, 137, 139, 551, 578],
    "EBIT": [983, 1181, 657, 2674, 3513],
    "EBIT Margin": [15.2, 17.0, 13.1, 12.8, 15.8],
    "Finance Cost": [51, 49, 56, 203, 197],
    "Profit before Tax": [932, 1132, 601, 2471, 3316],
    "Profit before Tax Margin": [14.4, 16.3, 12.0, 11.8, 14.9],
    "Tax": [242, 249, 158, 624, 798],
    "Profit After Tax": [690, 883, 443, 1847, 2518],
    "PAT Margin": [10.7, 12.7, 8.8, 8.8, 11.3],
    "EPS": [14.93, 18.85, 9.58, 39.98, 53.73]
}

margin_topics = {
    "EBITDA Margin",
    "EBIT Margin",
    "Profit before Tax Margin",
    "PAT Margin"
}

for topic, values in pnl_exact_values.items():
    unit = "percent" if topic in margin_topics else ("Rs." if topic == "EPS" else "Rs. Mn")

    for (_, reporting_period), value in zip(period_columns, values):
        add_record(
            "Profit and loss statement",
            topic,
            f"{topic} reported in the Q4 FY24 Profit & Loss Statement",
            value,
            unit,
            reporting_period,
            6
        )


pnl_change_values = [
    ("Revenue from Operations — YoY", "Year-on-year change", -7.0, "Q4 FY24 vs Q4 FY23"),
    ("Revenue from Operations — QoQ", "Quarter-on-quarter change", 28.8, "Q4 FY24 vs Q3 FY24"),
    ("Revenue from Operations — YoY", "Year-on-year change", -6.4, "FY24 vs FY23"),

    ("EBITDA — YoY", "Year-on-year change", -12.8, "Q4 FY24 vs Q4 FY23"),
    ("EBITDA — QoQ", "Quarter-on-quarter change", 54.6, "Q4 FY24 vs Q3 FY24"),
    ("EBITDA — YoY", "Year-on-year change", -22.8, "FY24 vs FY23"),

    ("EBIT — YoY", "Year-on-year change", -16.7, "Q4 FY24 vs Q4 FY23"),
    ("EBIT — QoQ", "Quarter-on-quarter change", 49.7, "Q4 FY24 vs Q3 FY24"),
    ("EBIT — YoY", "Year-on-year change", -23.9, "FY24 vs FY23"),

    ("Profit before Tax — YoY", "Year-on-year change", -17.7, "Q4 FY24 vs Q4 FY23"),
    ("Profit before Tax — QoQ", "Quarter-on-quarter change", 55.2, "Q4 FY24 vs Q3 FY24"),
    ("Profit before Tax — YoY", "Year-on-year change", -25.5, "FY24 vs FY23"),

    ("Profit After Tax — YoY", "Year-on-year change", -21.9, "Q4 FY24 vs Q4 FY23"),
    ("Profit After Tax — QoQ", "Quarter-on-quarter change", 55.8, "Q4 FY24 vs Q3 FY24"),
    ("Profit After Tax — YoY", "Year-on-year change", -26.7, "FY24 vs FY23")
]

for topic, description, value, period in pnl_change_values:
    add_record(
        "Profit and loss statement",
        topic,
        description,
        value,
        "percent",
        period,
        6
    )

add_record(
    "Profit and loss statement",
    "Marketing and sales promotion expense",
    "Marketing and sales promotion expense included in Other Expenses",
    120.4,
    "Rs. Mn",
    "Q4 FY24",
    6
)
add_record(
    "Profit and loss statement",
    "Comparative marketing and sales promotion expense",
    "Comparative marketing and sales promotion expense included in Other Expenses",
    175.3,
    "Rs. Mn",
    "Q3 FY24",
    6
)


# ------------------------------------------------------------------
# Company profile — page 8
# ------------------------------------------------------------------
company_profile_records = [
    ("Company history",
     "Period from which the company describes its history",
     1978, "year", "1978"),

    ("Market position",
     "Company described among India's renowned brands and marketers of fabrics, readymade garments and other textile products",
     "Amongst India’s most renowned brands and marketers",
     "text", "Q4 & FY24"),

    ("Product categories",
     "Product categories represented in the company profile",
     "Fabrics, readymade garments and other textile products",
     "text", "Q4 & FY24"),

    ("Brand portfolio",
     "Brands and sub-brands explicitly listed",
     "Siyaram, Mistair, J. Hampstead, CADINI and Oxemberg",
     "text", "Q4 & FY24"),

    ("Retail and online presence",
     "Commercial presence described in the company profile",
     "Franchises, retail stores and online platform presence",
     "text", "Q4 & FY24"),

    ("Manufacturing certifications",
     "Certifications printed in the company profile",
     "ISO 14001:2008 and 45,001",
     "text", "Q4 & FY24"),

    ("Manufacturing locations",
     "Locations of integrated manufacturing plants",
     "Tarapur, Daman, Amravati and Silvassa",
     "text", "Q4 & FY24"),

    ("Distribution ecosystem",
     "Distribution ecosystem described as focusing on all market segments",
     "Brands focusing on all the segments of the market",
     "text", "Q4 & FY24")
]

for topic, description, value, unit, period in company_profile_records:
    add_record(
        "Company profile",
        topic,
        description,
        value,
        unit,
        period,
        8
    )


# ------------------------------------------------------------------
# Corporate timeline — page 9
# ------------------------------------------------------------------
timeline_records = [
    ("Established", "Company established", "Established in 1978", "year", "1978–1987"),
    ("Public listing", "Company went public", "Went Public in 1980", "year", "1978–1987"),
    ("Tarapur capacity", "Set up capacity for manufacturing, weaving and processing at Tarapur", "Tarapur manufacturing, weaving and processing capacity", "text", "1978–1987"),

    ("Siyaram brand promotion", "Started brand promotion exercise with the printed tagline", "Coming Home to Siyaram’s", "text", "1991–2009"),
    ("Oxemberg", "Introduced Oxemberg to venture into readymade garments", "Oxemberg introduced", "text", "1991–2009"),
    ("J. Hampstead", "Launched J. Hampstead with 100% pure worsted suiting fabrics", "J. Hampstead launched", "text", "1991–2009"),
    ("Silvassa weaving capacity", "Started and expanded weaving capacity at Silvassa", "Silvassa weaving capacity", "text", "1991–2009"),
    ("Mistair", "Launched Mistair as a fashion brand for fabrics", "Mistair launched", "text", "1991–2009"),

    ("Most trusted brand", "Siyaram’s voted the most trusted brand by Economic Times and Nielsen Media Research", "Most trusted brand recognition", "text", "2013–2020"),
    ("Cadini", "Acquired the Italian Brand Cadini", "Cadini acquired", "text", "2013–2020"),
    ("Amravati unit", "Set up indigo rope dyeing unit at Amravati", "Amravati indigo rope dyeing unit", "text", "2013–2020"),
    ("Siyaram’s Mozzo", "Launched Siyaram’s Mozzo as a casual apparel brand", "Siyaram’s Mozzo launched", "text", "2013–2020"),
    ("Guinness World Record", "Set Guinness World Record for online Textile Mahakumbh", "Online Textile Mahakumbh Guinness World Record", "text", "2013–2020"),

    ("DEN-KNIT", "Launched DEN-KNIT knitted denim fabric brand", "DEN-KNIT launched", "text", "2021–2023"),
    ("Tessio", "Launched Siyaram’s Exclusive Knit Wear Brand Tessio", "Tessio launched", "text", "2021–2023"),
    ("EVITA & BREEZY", "Launched bamboo blended shirting fabric range under the EVITA and BREEZY sub-brands", "EVITA & BREEZY bamboo blended shirting range", "text", "2021–2023"),
    ("Ethnair", "Launched Ethnic wear fabric brand Ethnair", "Ethnair launched", "text", "2021–2023")
]

for topic, description, value, unit, period in timeline_records:
    add_record(
        "Corporate timeline",
        topic,
        description,
        value,
        unit,
        period,
        9
    )


# ------------------------------------------------------------------
# Operational footprint — page 10
# ------------------------------------------------------------------
operational_records = [
    ("Distributors",
     "Distributors spread across pin codes",
     "800+", "distributors", "Q4 & FY24"),

    ("Fabric sold",
     "Fabric sold in FY24; company estimate",
     "~100", "Mn meters", "FY24"),

    ("Stores across nation",
     "Stores across the nation; company estimate",
     "245+", "stores", "Q4 & FY24"),

    ("Retail space",
     "Retail space; company estimate",
     "~1.85", "L sqft", "Q4 & FY24"),

    ("Apparels sold",
     "Apparels sold in FY24; company estimate",
     "~4.5", "Mn pieces", "FY24"),

    ("Customers served",
     "Customers served; company estimate",
     "5 and counting", "Mn customers", "Q4 & FY24"),

    ("End markets",
     "Commercial channels and end markets listed on the page",
     "Distributors; MBO’s; Institutions; Online Marketplace; Exclusive Shops",
     "text", "Q4 & FY24")
]

for topic, description, value, unit, period in operational_records:
    add_record(
        "Operational footprint",
        topic,
        description,
        value,
        unit,
        period,
        10
    )


reference_values_df = pd.DataFrame(
    records,
    columns=REFERENCE_FIELDS
)

reference_values_df = (
    reference_values_df
    .astype(object)
    .where(
        pd.notna(reference_values_df),
        None
    )
)

reference_records = (
    reference_values_df
    .to_dict(
        orient="records"
    )
)

print(
    "Reference records:",
    len(reference_values_df)
)

display(
    reference_values_df.head(10)
)

Reference records: 199


,Category,Topic,Description,Value,Unit,Reporting Period,Source Location
0,Presentation metadata,Presentation title,Title of the investor presentation,Investor Presentation | Q4 & FY24,text,Q4 & FY24,PDF page 1
1,Presentation metadata,Company name,Company identified as the preparer of the pres...,Siyaram Silk Mills Limited,text,Q4 & FY24,PDF page 2
2,Presentation metadata,Safe Harbor,Presentation prepared solely for information p...,Safe Harbor statement,text,Q4 & FY24,PDF page 2
3,Management commentary,Market conditions,Market conditions faced by the company,Subdued consumer demand and challenging market...,text,Q4 FY24,PDF page 4
4,Management commentary,Revenue from Operations,Revenue from Operations reported in management...,6464,Rs. Mn,Q4 FY24,PDF page 4
5,Management commentary,Comparative Revenue from Operations,Comparative Revenue from Operations reported i...,6948,Rs. Mn,Q4 FY23,PDF page 4
6,Management commentary,Revenue mix — Fabric,Fabric share of the Q4 FY24 revenue mix,82,percent,Q4 FY24,PDF page 4
7,Management commentary,Revenue mix — Garments,Garments share of the Q4 FY24 revenue mix,13,percent,Q4 FY24,PDF page 4
8,Management commentary,Revenue mix — Yarn & Others,Yarn & Others share of the Q4 FY24 revenue mix,5,percent,Q4 FY24,PDF page 4
9,Management commentary,EBITDA,EBITDA reported in management commentary,1059,Rs. Mn,Q4 FY24,PDF page 4


In [13]:
# ============================================================
# 12. Reference record-count validation
# ============================================================

observed_category_counts = (
    reference_values_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


print(
    "Expected total:",
    EXPECTED_REFERENCE_RECORD_COUNT
)

print(
    "Observed total:",
    len(reference_values_df)
)

print(
    "Record count valid:",
    record_count_valid
)

print(
    "Category counts valid:",
    category_counts_valid
)

print(
    json.dumps(
        observed_category_counts,
        indent=2,
        ensure_ascii=False
    )
)


if not record_count_valid:
    raise AssertionError(
        "Unexpected D11 reference-record count."
    )

if not category_counts_valid:
    raise AssertionError(
        "Unexpected D11 category distribution."
    )

Expected total: 199
Observed total: 199
Record count valid: True
Category counts valid: True
{
  "Profit and loss statement": 102,
  "Quarterly business performance": 45,
  "Management commentary": 17,
  "Corporate timeline": 17,
  "Company profile": 8,
  "Operational footprint": 7,
  "Presentation metadata": 3
}


In [14]:
# ============================================================
# 13. Reference schema and field-type validation
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == REFERENCE_FIELDS
)


MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


type_issue_rows = []
missing_mandatory_rows = []


for row_index, row in (
    reference_values_df.iterrows()
):

    for field in MANDATORY_STRING_FIELDS:

        value = row[
            field
        ]

        if (
            value is None
            or str(value).strip() == ""
        ):
            missing_mandatory_rows.append({
                "Record Index":
                    int(row_index),

                "Field":
                    field
            })

        elif not isinstance(
            value,
            str
        ):
            type_issue_rows.append({
                "Record Index":
                    int(row_index),

                "Field":
                    field,

                "Observed Type":
                    type(value).__name__
            })


    value = row[
        "Value"
    ]

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(
                value,
                (
                    str,
                    int,
                    float
                )
            )
        )
    ):
        type_issue_rows.append({
            "Record Index":
                int(row_index),

            "Field":
                "Value",

            "Observed Type":
                type(value).__name__
        })


type_issues_df = pd.DataFrame(
    type_issue_rows
)

missing_mandatory_df = pd.DataFrame(
    missing_mandatory_rows
)


field_types_valid = (
    type_issues_df.empty
)

mandatory_fields_complete = (
    missing_mandatory_df.empty
)


print(
    "Reference schema valid:",
    reference_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)


if not reference_schema_valid:
    raise AssertionError(
        "D11 reference schema is invalid."
    )

if not field_types_valid:
    display(
        type_issues_df
    )

    raise AssertionError(
        "D11 reference field types are invalid."
    )

if not mandatory_fields_complete:
    display(
        missing_mandatory_df
    )

    raise AssertionError(
        "D11 mandatory reference fields are incomplete."
    )

Reference schema valid: True
Field types valid: True
Mandatory fields complete: True


In [15]:
# ============================================================
# 14. Duplicate, source-location and fidelity checks
# ============================================================

duplicate_mask = (
    reference_values_df
    .duplicated(
        subset=REFERENCE_FIELDS,
        keep=False
    )
)

duplicate_record_count = int(
    duplicate_mask.sum()
)


source_location_pattern_valid = bool(
    reference_values_df[
        "Source Location"
    ]
    .map(
        lambda value:
            bool(
                re.fullmatch(
                    r"PDF page (?:1|2|4|5|6|8|9|10)",
                    value
                )
            )
    )
    .all()
)


expected_source_pages = {
    "PDF page 1",
    "PDF page 2",
    "PDF page 4",
    "PDF page 5",
    "PDF page 6",
    "PDF page 8",
    "PDF page 9",
    "PDF page 10"
}

observed_source_pages = set(
    reference_values_df[
        "Source Location"
    ]
)

source_page_coverage_valid = (
    observed_source_pages
    == expected_source_pages
)


required_qualified_values = {
    "800+",
    "~100",
    "245+",
    "~1.85",
    "~4.5",
    "5 and counting"
}

observed_qualified_values = {
    value
    for value
    in reference_values_df[
        "Value"
    ]
    if (
        isinstance(
            value,
            str
        )
        and value
        in required_qualified_values
    )
}

qualified_values_valid = (
    observed_qualified_values
    == required_qualified_values
)


print(
    "Duplicate records:",
    duplicate_record_count
)

print(
    "Source-location format valid:",
    source_location_pattern_valid
)

print(
    "Source-page coverage valid:",
    source_page_coverage_valid
)

print(
    "Qualified values preserved:",
    qualified_values_valid
)


if duplicate_record_count != 0:

    display(
        reference_values_df.loc[
            duplicate_mask
        ]
    )

    raise AssertionError(
        "Unexpected duplicate D11 reference records."
    )


if not source_location_pattern_valid:
    raise AssertionError(
        "Invalid D11 source-location format."
    )


if not source_page_coverage_valid:
    raise AssertionError(
        "Unexpected D11 reference source-page coverage."
    )


if not qualified_values_valid:
    raise AssertionError(
        "One or more qualified D11 values were not "
        "preserved correctly."
    )

Duplicate records: 0
Source-location format valid: True
Source-page coverage valid: True
Qualified values preserved: True


In [16]:
# ============================================================
# 15. Indicator-level document assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Reading Order Quality",

        "Score":
            "Medium",

        "Evidence Source":
            "PDF text extraction + manual slide inspection",

        "Justification":
            "The presentation has a clear slide sequence, but several "
            "slides use independent spatial regions, stacked charts, "
            "timeline structures and infographic layouts whose "
            "relationships require structural interpretation."
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Table Structure Integrity",

        "Score":
            "High",

        "Evidence Source":
            "Chart and P&L table inspection + extracted-text inspection",

        "Justification":
            "Page 5 contains three stacked financial charts in which "
            "quarterly components must remain associated with fiscal "
            "years and annual totals, while page 6 contains a dense "
            "multi-period table with values, margins and change columns."
    },

    {
        "Dimension":
            "Structural Readiness",

        "Indicator":
            "Section/Header Hierarchy",

        "Score":
            "Low",

        "Evidence Source":
            "Manual presentation inspection",

        "Justification":
            "Slide titles and major presentation sections are explicit "
            "and consistently identifiable throughout the document."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Sharpness",

        "Score":
            "Low",

        "Evidence Source":
            "Manual visual inspection",

        "Justification":
            "The born-digital presentation is visually clear and "
            "relevant text, chart labels and numerical values are "
            "readable."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "Noise / Degradation",

        "Score":
            "Low",

        "Evidence Source":
            "Manual visual inspection",

        "Justification":
            "No relevant scanning noise, blur or visual degradation "
            "affects the document."
    },

    {
        "Dimension":
            "Visual/OCR Readiness",

        "Indicator":
            "OCR Dependency",

        "Score":
            "Low",

        "Evidence Source":
            "Automated PDF text extraction",

        "Justification":
            "The presentation contains a usable native text layer and "
            "OCR is not required for machine-readable text recovery."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Terminology Consistency",

        "Score":
            "Low",

        "Evidence Source":
            "Manual financial and corporate-content inspection",

        "Justification":
            "Financial terminology such as Revenue, EBITDA, EBIT, PAT, "
            "EPS and margins is used consistently, while corporate and "
            "operational terminology is also stable within its sections."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Schema Alignment",

        "Score":
            "Medium",

        "Evidence Source":
            "Reference-schema comparison",

        "Justification":
            "The common schema accommodates the selected content, but "
            "it must represent several semantic record types including "
            "financial observations, qualitative management statements, "
            "company-profile facts, timeline events and operational KPIs."
    },

    {
        "Dimension":
            "Semantic Quality",

        "Indicator":
            "Numerical Density",

        "Score":
            "High",

        "Evidence Source":
            "Automated text profiling + manual inspection",

        "Justification":
            "A substantial portion of the relevant presentation "
            "contains financial amounts, percentages, quarterly and "
            "annual values, margin measures and operational KPIs."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Required Field Presence",

        "Score":
            "Low",

        "Evidence Source":
            "Reference-value verification",

        "Justification":
            "All information required by the predefined 199-record "
            "extraction scope is explicitly represented in the source."
    },

    {
        "Dimension":
            "Completeness and Consistency",

        "Indicator":
            "Internal Consistency",

        "Score":
            "Medium",

        "Evidence Source":
            "Manual source and reference inspection",

        "Justification":
            "Financial observations are generally coherent across "
            "commentary, charts and the P&L table, but repeated metrics "
            "and differently scoped representations, including separate "
            "store-count observations, require context-preserving "
            "comparison rather than automatic reconciliation."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Format Heterogeneity",

        "Score":
            "High",

        "Evidence Source":
            "Document profiling + manual layout inspection",

        "Justification":
            "The presentation combines narrative commentary, stacked "
            "financial charts, a dense financial table, company-profile "
            "text, a multi-phase timeline, photographs and infographic "
            "operational metrics."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",

        "Indicator":
            "Unit / Label Variability",

        "Score":
            "High",

        "Evidence Source":
            "Reference and source inspection",

        "Justification":
            "The extraction scope combines Rs. Mn, Rs. crores, "
            "percentages, rupees per share, stores, distributors, "
            "million metres, million pieces, retail-space measures, "
            "historical intervals and qualified values using ~, + "
            "and 'and counting'."
    }
]


indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(
    indicator_assessment_df
)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Medium,PDF text extraction + manual slide inspection,"The presentation has a clear slide sequence, b..."
1,Structural Readiness,Table Structure Integrity,High,Chart and P&L table inspection + extracted-tex...,Page 5 contains three stacked financial charts...
2,Structural Readiness,Section/Header Hierarchy,Low,Manual presentation inspection,Slide titles and major presentation sections a...
3,Visual/OCR Readiness,Sharpness,Low,Manual visual inspection,The born-digital presentation is visually clea...
4,Visual/OCR Readiness,Noise / Degradation,Low,Manual visual inspection,"No relevant scanning noise, blur or visual deg..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated PDF text extraction,The presentation contains a usable native text...
6,Semantic Quality,Terminology Consistency,Low,Manual financial and corporate-content inspection,"Financial terminology such as Revenue, EBITDA,..."
7,Semantic Quality,Schema Alignment,Medium,Reference-schema comparison,The common schema accommodates the selected co...
8,Semantic Quality,Numerical Density,High,Automated text profiling + manual inspection,A substantial portion of the relevant presenta...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All information required by the predefined 199...


In [17]:
# ============================================================
# 16. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}


expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}


invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)


observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)


missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)


if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )

if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )

if (
    len(indicator_assessment_df)
    != len(expected_indicators)
):
    raise ValueError(
        "Duplicate indicator rows detected."
    )


print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [18]:
# ============================================================
# 17. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}


indicator_assessment_df[
    "Numeric Score"
] = (
    indicator_assessment_df[
        "Score"
    ]
    .map(
        score_to_numeric
    )
)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),

        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(
    mean_score
):

    if mean_score < 1.5:
        return "Low"

    elif mean_score < 2.5:
        return "Medium"

    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = (
    dimension_assessment_df[
        "Mean_Score"
    ]
    .apply(
        classify_dimension_score
    )
)


dimension_assessment_df[
    "Mean_Score"
] = (
    dimension_assessment_df[
        "Mean_Score"
    ]
    .round(2)
)


display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.5,2,Medium
1,Representation and Normalisation Complexity,3.0,2,High
2,Semantic Quality,2.0,3,Medium
3,Structural Readiness,2.0,3,Medium
4,Visual/OCR Readiness,1.0,3,Low


In [19]:
# ============================================================
# 18. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            (
                "Arithmetic mean of indicator scores "
                "within each dimension."
            ),

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [20]:
# ============================================================
# 19. D11-specific reference-semantic integrity checks
# ============================================================

timeline_year_unit_checks = {
    "established_unit_year": bool(
        reference_values_df.loc[
            (reference_values_df["Category"] == "Corporate timeline")
            & (reference_values_df["Topic"] == "Established"),
            "Unit",
        ].eq("year").all()
    ),

    "public_listing_unit_year": bool(
        reference_values_df.loc[
            (reference_values_df["Category"] == "Corporate timeline")
            & (reference_values_df["Topic"] == "Public listing"),
            "Unit",
        ].eq("year").all()
    ),
}

timeline_year_units_valid = all(
    timeline_year_unit_checks.values()
)

print(
    json.dumps(
        timeline_year_unit_checks,
        indent=2,
        ensure_ascii=False
    )
)

print(
    "Timeline year units valid:",
    timeline_year_units_valid
)

{
  "established_unit_year": true,
  "public_listing_unit_year": true
}
Timeline year units valid: True


In [28]:
# ============================================================
# 20. Reference-construction metadata
# ============================================================

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "source_format":
        SOURCE_FORMAT,

    "reference_file":
        "D11_reference_values.csv",

    "reference_construction_method":
        (
            "Manual document-grounded construction "
            "followed by programmatic integrity checks"
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        REFERENCE_FIELDS,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "visual_chart_estimation_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_value_repair_applied":
        False,

    "qualified_values_preserved":
        True,

    "repeated_cross_section_values_preserved":
        True,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

    "notes":
        (
            "D11 combines narrative commentary, financial charts, "
            "a multi-period P&L table, company-profile content, "
            "timeline events and operational infographic metrics. "
            "Repeated values are preserved by source context and "
            "qualified values retain their original notation. "
            "A reference-integrity review corrected the Unit field "
            "for the 'Established' and 'Public listing' timeline "
            "records to 'year', consistent with the source and "
            "reference-field semantics."
        )
}

print(
    json.dumps(
        REFERENCE_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "D11",
  "document_name": "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "source_file_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "source_format": "PDF",
  "reference_file": "D11_reference_values.csv",
  "reference_construction_method": "Manual document-grounded construction followed by programmatic integrity checks",
  "expected_reference_record_count": 199,
  "observed_reference_record_count": 199,
  "reference_fields": [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
  ],
  "manual_calculation_applied": false,
  "semantic_inference_applied": false,
  "visual_chart_estimation_applied": false,
  "unit_conversion_applied": false,
  "source_value_repair_applied": false,
  "qualified_values_preserved": true,
  "repeated_cross_section_values_preserved": true,
  "reference_values_branch_

In [29]:
# ============================================================
# 21. Final reference-integrity conclusion
# ============================================================

REFERENCE_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "reference_record_count":
        int(
            len(reference_values_df)
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "category_counts":
        observed_category_counts,

    "reference_fields":
        REFERENCE_FIELDS,

    "source_pages_represented":
        sorted(
            reference_values_df[
                "Source Location"
            ]
            .str.extract(
                r"PDF page (\d+)",
                expand=False
            )
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        ),

    "qualified_values_preserved":
        bool(
            qualified_values_valid
        ),

    "timeline_year_units_valid":
        bool(
            timeline_year_units_valid
        ),

    "branch_independent_reference":
        True,
}

In [30]:
# ============================================================
# 22. Output paths
# ============================================================

REFERENCE_INTEGRITY_PASSED = all([
    PAGE_COUNT_VALID,
    TEXT_EXTRACTABLE,
    reference_schema_valid,
    record_count_valid,
    category_counts_valid,
    field_types_valid,
    mandatory_fields_complete,
    duplicate_record_count == 0,
    source_location_pattern_valid,
    source_page_coverage_valid,
    qualified_values_valid,
    timeline_year_units_valid
])


REFERENCE_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "page_count_valid":
        bool(
            PAGE_COUNT_VALID
        ),

    "text_extractable":
        bool(
            TEXT_EXTRACTABLE
        ),

    "reference_schema_valid":
        bool(
            reference_schema_valid
        ),

    "record_count_valid":
        bool(
            record_count_valid
        ),

    "category_counts_valid":
        bool(
            category_counts_valid
        ),

    "field_types_valid":
        bool(
            field_types_valid
        ),

    "mandatory_fields_complete":
        bool(
            mandatory_fields_complete
        ),

    "duplicate_record_count":
        int(
            duplicate_record_count
        ),

    "source_location_pattern_valid":
        bool(
            source_location_pattern_valid
        ),

    "source_page_coverage_valid":
        bool(
            source_page_coverage_valid
        ),

    "qualified_values_preserved":
        bool(
            qualified_values_valid
        ),

    "timeline_year_unit_checks":
        timeline_year_unit_checks,

    "timeline_year_units_valid":
        bool(
            timeline_year_units_valid
        ),

    "manual_reference_construction":
        True,

    "calculation_applied":
        False,

    "inference_applied":
        False,

    "visual_estimation_applied":
        False,

    "reference_integrity_passed":
        bool(
            REFERENCE_INTEGRITY_PASSED
        )
}


print(
    json.dumps(
        REFERENCE_INTEGRITY,
        indent=2,
        ensure_ascii=False
    )
)


if not REFERENCE_INTEGRITY_PASSED:

    raise AssertionError(
        "D11 reference-integrity checks failed."
    )

{
  "document_id": "D11",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "source_file_sha256": "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952",
  "page_count_valid": true,
  "text_extractable": true,
  "reference_schema_valid": true,
  "record_count_valid": true,
  "category_counts_valid": true,
  "field_types_valid": true,
  "mandatory_fields_complete": true,
  "duplicate_record_count": 0,
  "source_location_pattern_valid": true,
  "source_page_coverage_valid": true,
  "qualified_values_preserved": true,
  "timeline_year_unit_checks": {
    "established_unit_year": true,
    "public_listing_unit_year": true
  },
  "timeline_year_units_valid": true,
  "manual_reference_construction": true,
  "calculation_applied": false,
  "inference_applied": false,
  "visual_estimation_applied": false,
  "reference_integrity_passed": true
}


In [31]:
# ============================================================
# 23. Export Stage 1 outputs
# ============================================================

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR /
    "D11_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR /
    "D11_reference_values.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D11_extraction_schema.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR /
    "D11_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR /
    "D11_reference_summary.json"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR /
    "D11_reference_metadata.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR /
    "D11_reference_integrity.json"
)

DOCUMENT_METADATA_PATH = (
    OUTPUT_DIR /
    "D11_document_metadata.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR /
    "D11_document_characterisation.json"
)

PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR /
    "D11_page_characterisation.csv"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D11_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR /
    "D11_dimension_assessment.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR /
    "D11_quality_evidence.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR /
    "D11_extraction_task.txt"
)

In [32]:
# ============================================================
# 24. Final notebook summary
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)


REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_records,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)


page_characterisation_df.to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)


EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8"
)


json_outputs = [
    (
        EXTRACTION_SCHEMA_PATH,
        EXTRACTION_SCHEMA
    ),

    (
        REFERENCE_SCHEMA_PATH,
        REFERENCE_SCHEMA
    ),

    (
        REFERENCE_SUMMARY_PATH,
        REFERENCE_SUMMARY
    ),

    (
        REFERENCE_METADATA_PATH,
        REFERENCE_METADATA
    ),

    (
        REFERENCE_INTEGRITY_PATH,
        REFERENCE_INTEGRITY
    ),

    (
        DOCUMENT_METADATA_PATH,
        DOCUMENT_METADATA
    ),

    (
        DOCUMENT_CHARACTERISATION_PATH,
        DOCUMENT_CHARACTERISATION
    ),

    (
        QUALITY_EVIDENCE_PATH,
        QUALITY_EVIDENCE
    )
]


for output_path, content in (
    json_outputs
):

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            content,
            file,
            indent=2,
            ensure_ascii=False
        )


print(
    "D11 Stage 1 outputs exported successfully."
)

D11 Stage 1 outputs exported successfully.


In [33]:
# ============================================================
# 25. Final checks and output listing
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    EXTRACTION_SCHEMA_PATH,
    REFERENCE_SCHEMA_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    REFERENCE_INTEGRITY_PATH,
    DOCUMENT_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    PAGE_CHARACTERISATION_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,
    EXTRACTION_TASK_PATH
]


all_outputs_exist = all(
    path.exists()
    for path
    in GENERATED_OUTPUTS
)


FINAL_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "source_file":
        SOURCE_PATH.name,

    "physical_page_count":
        PAGE_COUNT,

    "reference_record_count":
        int(
            len(reference_values_df)
        ),

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "category_counts":
        observed_category_counts,

    "reference_integrity_passed":
        bool(
            REFERENCE_INTEGRITY_PASSED
        ),

    "all_outputs_exist":
        bool(
            all_outputs_exist
        ),

    "outputs_created": [
        path.name
        for path
        in GENERATED_OUTPUTS
    ]
}


print(
    json.dumps(
        FINAL_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)


if not all([
    REFERENCE_INTEGRITY_PASSED,
    all_outputs_exist
]):

    raise AssertionError(
        "D11 Stage 1 notebook did not complete successfully."
    )


print(
    "\nD11 Stage 1 completed successfully."
)

print(
    "Next step: D11 Branch A — direct PDF ingestion."
)

{
  "document_id": "D11",
  "source_file": "D11 - Investor-Presentation-May-2024.pdf",
  "physical_page_count": 10,
  "reference_record_count": 199,
  "expected_reference_record_count": 199,
  "category_counts": {
    "Profit and loss statement": 102,
    "Quarterly business performance": 45,
    "Management commentary": 17,
    "Corporate timeline": 17,
    "Company profile": 8,
    "Operational footprint": 7,
    "Presentation metadata": 3
  },
  "reference_integrity_passed": true,
  "all_outputs_exist": true,
  "outputs_created": [
    "D11_reference_values.csv",
    "D11_reference_values.json",
    "D11_extraction_schema.json",
    "D11_reference_schema.json",
    "D11_reference_summary.json",
    "D11_reference_metadata.json",
    "D11_reference_integrity.json",
    "D11_document_metadata.json",
    "D11_document_characterisation.json",
    "D11_page_characterisation.csv",
    "D11_indicator_assessment.csv",
    "D11_dimension_assessment.csv",
    "D11_quality_evidence.json",
    

In [34]:
# ============================================================
# 26. Download generated outputs
# ============================================================

for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            str(
                output_path
            )
        )

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>